# Inspect Pipeline Monitoring

In [1]:
from pyspark.sql import SparkSession, functions as F

spark = (
    SparkSession.builder
    .master("spark://spark-master:7077")
    .appName("InspectPipelineMonitoring")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")
spark.conf.set("spark.sql.session.timeZone", "UTC")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/09/27 16:54:13 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [2]:
PIPELINE_EXECUTIONS_PATH = (
    "s3a://delta-data/metadata/pipeline_executions"
)

executions = spark.read.format("delta").load(PIPELINE_EXECUTIONS_PATH)
executions.printSchema()
executions.orderBy(F.desc("started_at")).show(100, truncate=False)

26/09/27 16:54:16 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties
SLF4J: Failed to load class "org.slf4j.impl.StaticLoggerBinder".
SLF4J: Defaulting to no-operation (NOP) logger implementation
SLF4J: See http://www.slf4j.org/codes.html#StaticLoggerBinder for further details.


root
 |-- pipeline_run_id: string (nullable = true)
 |-- execution_id: string (nullable = true)
 |-- stage: string (nullable = true)
 |-- dataset: string (nullable = true)
 |-- schema_version: string (nullable = true)
 |-- started_at: timestamp (nullable = true)
 |-- finished_at: timestamp (nullable = true)
 |-- duration_seconds: double (nullable = true)
 |-- processed_records: long (nullable = true)
 |-- inserted_records: long (nullable = true)
 |-- rejected_records: long (nullable = true)
 |-- validation_failure_count: long (nullable = true)



26/09/27 16:54:18 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


+------------------------------------+------------------------------------+-----------------+--------------------------------+--------------+--------------------------+--------------------------+----------------+-----------------+----------------+----------------+------------------------+
|pipeline_run_id                     |execution_id                        |stage            |dataset                         |schema_version|started_at                |finished_at               |duration_seconds|processed_records|inserted_records|rejected_records|validation_failure_count|
+------------------------------------+------------------------------------+-----------------+--------------------------------+--------------+--------------------------+--------------------------+----------------+-----------------+----------------+----------------+------------------------+
|b2d2466b-ae95-4bfe-ade1-95d96c1b00c0|9e7a464a-f752-465e-be04-d51c5764f6e8|analytics_refresh|zone_weather_demand_variation   |1.1 

In [3]:
spark.stop()